# Manchu CRNN: 60,000-image GPU experiment

This uses tyotakuki's generated images and lowercase romanization labels. Training covers 45,000 word forms in 60,000 images. It retains the small-sample fixes: ink cropping, height 32, original orientation, original noisy pixels, and group normalization in the CNN. It starts from fresh weights, not the 16-word checkpoint.

Choose **Runtime → Change runtime type → GPU** before running. The first run copies the 15.7 GB source ZIP to temporary Colab storage and preprocesses only the 61,500 selected training/development images. This preparation can take several minutes; the source data is read-only. The 250 final-test images are not staged or evaluated.

Checkpoints, configuration and history save under this dedicated Drive folder. Colab caches disappear when the runtime resets; rerunning staging rebuilds them. A session interrupted mid-epoch resumes from the last completed epoch, so at most one epoch of progress is lost. This is a new protocol, not an exact reproduction of Chung and Choi.

In [ ]:
from pathlib import Path
import sys, json
from google.colab import drive

drive.mount("/content/drive")
THESIS = Path("/content/drive/MyDrive/4 Senior/Thesis")
PACKAGE = THESIS / "Manchu-CRNN-60k"
assert PACKAGE.is_dir(), f"Folder not found: {PACKAGE}; check the Thesis path and Drive sync."
sys.path.insert(0, str(PACKAGE))
import torch
import crnn60k_runtime as training

assert torch.cuda.is_available(), "Choose a GPU runtime before training."
DEVICE = torch.device("cuda")
torch.set_num_threads(2)
print("GPU:", torch.cuda.get_device_name())
print("PyTorch:", torch.__version__)

## Configuration and split verification

The 1,000-image seen-word development set tests new images of familiar forms. The 500-image unseen-word development set tests forms excluded from training and selects the checkpoint by character error rate. The 250-image unseen-word final test remains reserved. Font identities are unavailable, so this is not a verified font-disjoint experiment.

Use a new RUN_ID for changed settings. With RESUME=True, an existing run continues from its last completed epoch; configuration changes are rejected. The initial schedule allows 20 epochs with early stopping after at least 8 epochs and 5 epochs without improvement. The timing estimate below excludes validation and checkpoint overhead.

In [ ]:
RUN_ID = "crnn_60k_v1"
EPOCHS = 20
BATCH_SIZE = 16
WORKERS = 2
RESUME = True
COPY_ARCHIVE_TO_LOCAL = True  # Avoid thousands of random reads over mounted Drive.
RUN_TRAINING = False  # Set True after reviewing the benchmark and sample gallery.
CACHE = Path("/content/manchu60k")
RUN = PACKAGE / "runs" / RUN_ID

protocol, splits = training.load_protocol(PACKAGE)
codec = {char: index+1 for index, char in enumerate(protocol["alphabet"])}
assert len(splits["train"]) == 60000
print(json.dumps(protocol["counts"], indent=2))
print("Distinct training forms:", len({r["romanization"] for r in splits["train"]}))
print("Outputs:", RUN)

## Stage data and check every CTC alignment

The source archive is copied once to local Colab disk. Selected members are checked against their recorded sizes/CRCs, then cropped and resized into local PNGs. No full extraction occurs and no original Drive files are modified. Staging verifies that the CTC timeline can fit every label. If preflight fails, stop and inspect its report rather than dropping bad samples or hiding infinite losses.

In [ ]:
source = THESIS / protocol["source_archive"]
archive = training.copy_archive(source, CACHE, protocol["source_archive_bytes"]) if COPY_ARCHIVE_TO_LOCAL else source
PROCESSED = training.prepare_cache(archive, CACHE, splits)
print("Processed images:", PROCESSED)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(4, 2, figsize=(14, 8))
for axis, row in zip(axes.flat, splits["train"][:8]):
    with Image.open(PROCESSED / Path(row["member"]).with_suffix(".png")) as image:
        axis.imshow(image, cmap="gray")
    axis.set_title(row["romanization"])
    axis.axis("off")
plt.tight_layout()

## Benchmark the actual GPU

This performs 23 training updates on a disposable model (3 warm-up plus 20 timed), then discards it. The real experiment starts fresh. The CNN, LSTM, and CTC loss use CUDA. The CNN still processes each image individually to preserve the validated treatment of variable widths, so GPU utilization will not be perfect. Input preprocessing is already cached locally.

In [ ]:
batches = training.loader(splits["train"], PROCESSED, codec, BATCH_SIZE, WORKERS, True)
timing = training.benchmark(batches, DEVICE, codec)
print(f"Approximate {EPOCHS}-epoch training time: {timing['estimated_epoch_minutes']*EPOCHS/60:.1f} hours, plus validation/checkpoints.")

## Train and save

Set RUN_TRAINING=True above, then run this cell. Progress prints every 100 batches and after each epoch. Best.pt is selected by unseen-word development CER; last.pt also stores optimizer, history and random-generator state for completed-epoch resume. Test data is never used here. Watch loss, decoded examples, and both development sets; small-sample success does not guarantee broad recognition.

In [ ]:
if RUN_TRAINING:
    RUN.mkdir(parents=True, exist_ok=True)
    training.write_json(RUN / "benchmark.json", timing)
    history = training.train(PACKAGE, PROCESSED, splits, codec, RUN, DEVICE,
                             epochs=EPOCHS, batch_size=BATCH_SIZE, workers=WORKERS, resume=RESUME)
else:
    print("Training is off. Review the benchmark, then set RUN_TRAINING=True.")

## Review the selected model

These remain development results. Reserve the final test until the model and training procedure are fixed. Keep tyotakuki's authorship and dataset provenance in every report.

In [ ]:
if (RUN / "best.pt").exists():
    from crnn60k_model import load_model
    model, checkpoint = load_model(RUN / "best.pt")
    model = model.to(DEVICE)
    print("Selected epoch:", checkpoint["epoch"])
    for name in ("val_seen_words", "val_unseen_words"):
        results = training.evaluate(model, training.loader(splits[name], PROCESSED, codec, BATCH_SIZE, WORKERS), DEVICE, codec)
        print(name, json.dumps(results, ensure_ascii=False, indent=2))
else:
    print("No completed training checkpoint yet.")